Bài tập Môn học Trí Tuệ Nhân Tạo - EE3063
Đề tài: Robot Localization 1D sử dụng Particle Filter

Nhóm Thực Hiện: [Điền tên thành viên nhóm]
MSSV: [Điền MSSV thành viên nhóm]

Nội dung:
1. Khai báo các tham số của bài toán.
2. Mô phỏng bước dự đoán (prediction) của Particle Filter.
3. Mô phỏng bước cập nhật trọng số (weighting) của Particle Filter.
4. Thực hiện bước tái lấy mẫu (resampling).
5. In kết quả các particle và trọng số.

In [11]:
# -*- coding: utf-8 -*-

import numpy as np
import math # Sử dụng cho math.pi và math.exp nếu không dùng numpy

# ==============================================================================
# Phần 1: Khai báo các tham số và giá trị cho trước của bài toán
# ==============================================================================
# Giải thích:
# Các giá trị này được lấy từ đề bài trong file "EE3063 - homework - 242.docx".

# Số lượng particle
M = 20

# Thời điểm trước đó (t-1)
x_t_minus_1 = 2.0  # Vị trí của robot ở thời điểm t-1 (m)
# Giả định tất cả các particle ở thời điểm t-1 có cùng vị trí này
particles_t_minus_1 = np.full(M, x_t_minus_1)

# Mô hình xử lý (Process Model)
V_t = 5.0       # Vận tốc danh định của robot ở thời điểm t (m/s)
delta_t = 0.2   # Khoảng thời gian (s)
sigma_V = 0.8   # Độ lệch chuẩn của nhiễu vận tốc delta_Vt

# Mô hình đo lường (Measurement Model)
x_L = 10.0      # Tọa độ của landmark (m)
d_measured = 7.1 # Khoảng cách đo được từ robot đến landmark tại thời điểm t (m)
sigma_d = 0.1   # Độ lệch chuẩn của nhiễu đo lường

    Tính giá trị hàm mật độ xác suất của phân bố chuẩn.
    Args:
        x (float): Giá trị cần tính PDF.
        mu (float): Giá trị trung bình của phân bố.
        sigma (float): Độ lệch chuẩn của phân bố.
    Returns:
        float: Giá trị PDF tại x.

In [12]:
# Hàm mật độ xác suất của phân bố chuẩn (Gaussian PDF)
# Công thức: (1 / (sigma * sqrt(2*pi))) * exp(-0.5 * ((x - mu) / sigma)^2)
def gaussian_pdf(x, mu, sigma):
    """

    """
    if sigma == 0: # Tránh lỗi chia cho 0
        return 1.0 if x == mu else 0.0
    coefficient = 1.0 / (sigma * np.sqrt(2 * np.pi))
    exponent = -0.5 * ((x - mu) / sigma)**2
    return coefficient * np.exp(exponent)

Yêu cầu: Nhóm cần tự tạo các số ngẫu nhiên này từ các trang web được chỉ định trong đề bài
và điền vào các danh sách dưới đây.

1. Số ngẫu nhiên cho bước Prediction (tạo nhiễu delta_Vt)
   - Tạo 20 số từ phân bố chuẩn với mean = 0, standard deviation = sigma_V = 0.8
   - Từ trang: https://www.socscistatistics.com/utilities/normaldistribution/default.aspx
   - Điền các giá trị vào danh sách `delta_V_samples`
Ví dụ (CẦN THAY THẾ BẰNG SỐ LIỆU THỰC TẾ TỪ WEB):

In [13]:
delta_V_samples_placeholder = [
    0.1, -0.2, 0.3, -0.4, 0.5, -0.6, 0.7, -0.8, 0.9, -1.0,
    1.1, -1.2, 0.15, -0.25, 0.35, -0.45, 0.55, -0.65, 0.75, -0.85
]

KIỂM TRA LẠI: Đảm bảo các bạn nhập đúng 20 số ngẫu nhiên đã tạo.
Các số này nên có giá trị xung quanh 0, với độ lệch chuẩn khoảng 0.8.

2. Số ngẫu nhiên cho bước Resampling
   - Tạo 20 số ngẫu nhiên từ phân bố đều trong khoảng [0, 1)
   - Từ trang: https://pinetools.com/random-number-generator
   - Điền các giá trị vào danh sách `resampling_uniform_samples`
Ví dụ (CẦN THAY THẾ BẰNG SỐ LIỆU THỰC TẾ TỪ WEB):

In [14]:
resampling_uniform_samples_placeholder = [
    0.05, 0.12, 0.18, 0.23, 0.29, 0.33, 0.39, 0.45, 0.51, 0.57,
    0.62, 0.68, 0.73, 0.79, 0.84, 0.88, 0.91, 0.95, 0.98, 0.01
]
# KIỂM TRA LẠI: Đảm bảo các bạn nhập đúng 20 số ngẫu nhiên (từ 0 đến dưới 1).

# Sử dụng placeholder nếu chưa có số liệu thực tế từ web
# LƯU Ý QUAN TRỌNG: KHI NỘP BÀI, HÃY THAY THẾ CÁC DANH SÁCH PLACEHOLDER NÀY
# BẰNG CÁC SỐ LIỆU THỰC TẾ MÀ NHÓM TẠO RA TỪ WEB.
delta_V_samples = np.array(delta_V_samples_placeholder)
resampling_uniform_samples = np.array(resampling_uniform_samples_placeholder)

if len(delta_V_samples) != M:
    raise ValueError(f"Cần {M} mẫu cho delta_V_samples, hiện tại có {len(delta_V_samples)}")
if len(resampling_uniform_samples) != M:
    raise ValueError(f"Cần {M} mẫu cho resampling_uniform_samples, hiện tại có {len(resampling_uniform_samples)}")


Phần 2: Bước Dự đoán (Prediction) và Cập nhật Trọng số (Weighting)

Giải thích:
- Bước Prediction: Từ vị trí các particle ở thời điểm t-1, dự đoán vị trí mới ở thời điểm t
  dựa trên mô hình chuyển động và thêm nhiễu.
  Công thức: x_t = x_{t-1} + (V_t + delta_Vt) * delta_t
- Bước Weighting: Tính trọng số cho mỗi particle dựa trên sự phù hợp của vị trí dự đoán
  với phép đo thực tế (d_measured).
  Trọng số w_t = p(d_measured | x_t), được tính bằng Gaussian PDF.

In [15]:
particles_t_predicted = np.zeros(M) # Lưu vị trí dự đoán của các particle
weights_t = np.zeros(M)             # Lưu trọng số của các particle

print("="*30)
print("BƯỚC DỰ ĐOÁN VÀ CẬP NHẬT TRỌNG SỐ")
print("="*30)
print(f"{'Particle (m)':<15} | {'x_t-1':<10} | {'delta_Vt':<10} | {'x_t_predicted':<15} | {'d_expected':<15} | {'Weight w_t':<15}")
print("-"*90)

for m in range(M):
    # 2.1. Bước Dự đoán (Prediction)
    # Lấy mẫu delta_Vt từ danh sách đã cung cấp (tạo từ web)
    delta_Vt_m = delta_V_samples[m]
    
    # Tính vị trí dự đoán cho particle m
    # x_t = x_{t-1} + V_t*delta_t + delta_Vt*delta_t
    # Hoặc x_t = x_{t-1} + (V_t + delta_Vt) * delta_t
    # Theo slide Uncertainty, trang 13: p(xt|xt-1, ut). ut ở đây là V_t*delta_t (quãng đường danh định)
    # và nhiễu được thêm vào.
    # xt = xt-1 + ut + noise
    # xt[m] = particles_t_minus_1[m] + V_t * delta_t + delta_Vt_m * delta_t (nếu delta_Vt_m là nhiễu vận tốc)
    # Hoặc xt[m] = particles_t_minus_1[m] + V_t * delta_t + sample_displacement_noise (nếu delta_Vt_m là nhiễu quãng đường)
    # Đề bài ghi: xt = xt-1 + Vt + delta_Vt*delta_t. Điều này hơi lạ về đơn vị nếu Vt là vận tốc.
    # Giả sử đề bài muốn nói: xt = xt-1 + (Vt_command + delta_Vt_noise) * delta_t
    # Hoặc xt = xt-1 + displacement_command + displacement_noise
    # Dựa theo process model "xt=xt-1+Vt+ΔVtΔt", nếu Vt là vận tốc, ΔVtΔt cũng là vận tốc * thời gian = quãng đường.
    # Vậy thì Vt trong công thức này phải là quãng đường đi được trong delta_t do vận tốc danh định.
    # Tức là Vt trong công thức của đề có thể hiểu là V_nominal * delta_t.
    # Và ΔVt trong công thức của đề có thể hiểu là delta_V_noise (nhiễu vận tốc).
    # Vậy: x_t[m] = x_{t-1}[m] + V_nominal * delta_t + delta_V_noise[m] * delta_t
    # Hay: x_t[m] = x_{t-1}[m] + (V_nominal + delta_V_noise[m]) * delta_t
    
    particles_t_predicted[m] = particles_t_minus_1[m] + (V_t + delta_Vt_m) * delta_t

    # 2.2. Bước Cập nhật Trọng số (Weighting)
    # Tính khoảng cách dự kiến từ particle m đến landmark
    d_expected_m = np.abs(x_L - particles_t_predicted[m])
    
    # Tính trọng số cho particle m dựa trên phép đo d_measured
    # Trọng số là giá trị PDF của N(d_expected_m, sigma_d^2) tại điểm d_measured
    weights_t[m] = gaussian_pdf(d_measured, d_expected_m, sigma_d)
    
    print(f"{m+1:<15} | {particles_t_minus_1[m]:<10.2f} | {delta_Vt_m:<10.2f} | {particles_t_predicted[m]:<15.4f} | {d_expected_m:<15.4f} | {weights_t[m]:<15.6e}")

# Chuẩn hóa trọng số (để tổng các trọng số bằng 1)
# Điều này cần thiết cho bước tái lấy mẫu
sum_weights = np.sum(weights_t)
if sum_weights == 0:
    print("\nCảnh báo: Tổng các trọng số bằng 0. Có thể tất cả các particle đều ở quá xa so với phép đo.")
    # Gán trọng số đều nếu tất cả bằng 0 để tránh lỗi chia cho 0 khi resampling
    normalized_weights_t = np.full(M, 1.0/M)
else:
    normalized_weights_t = weights_t / sum_weights

print("\nTrọng số đã chuẩn hóa:")
print(normalized_weights_t)


BƯỚC DỰ ĐOÁN VÀ CẬP NHẬT TRỌNG SỐ
Particle (m)    | x_t-1      | delta_Vt   | x_t_predicted   | d_expected      | Weight w_t     
------------------------------------------------------------------------------------------
1               | 2.00       | 0.10       | 3.0200          | 6.9800          | 1.941861e+00   
2               | 2.00       | -0.20      | 2.9600          | 7.0400          | 3.332246e+00   
3               | 2.00       | 0.30       | 3.0600          | 6.9400          | 1.109208e+00   
4               | 2.00       | -0.40      | 2.9200          | 7.0800          | 3.910427e+00   
5               | 2.00       | 0.50       | 3.1000          | 6.9000          | 5.399097e-01   
6               | 2.00       | -0.60      | 2.8800          | 7.1200          | 3.910427e+00   
7               | 2.00       | 0.70       | 3.1400          | 6.8600          | 2.239453e-01   
8               | 2.00       | -0.80      | 2.8400          | 7.1600          | 3.332246e+00   
9          

## Phần 3: Bước Tái lấy mẫu (Resampling)

Giải thích:
Bước này nhằm loại bỏ các particle có trọng số thấp và nhân bản các particle có trọng số cao.
Chúng ta sẽ sử dụng phương pháp Roulette Wheel Selection (chọn bánh xe quay).
1. Tính hàm phân phối tích lũy (CDF) của các trọng số đã chuẩn hóa.
2. Với mỗi particle mới cần tạo:
   a. Tạo một số ngẫu nhiên r trong khoảng [0, 1) (từ danh sách resampling_uniform_samples).
   b. Tìm particle đầu tiên trong tập cũ mà CDF của nó >= r.
   c. Sao chép particle đó vào tập particle mới.

In [16]:
particles_t_resampled = np.zeros(M) # Lưu vị trí của các particle sau khi tái lấy mẫu

# Tính CDF của các trọng số đã chuẩn hóa
cdf_weights = np.cumsum(normalized_weights_t)

print("\n="*30)
print("BƯỚC TÁI LẤY MẪU (RESAMPLING)")
print("="*30)
print(f"{'Particle mới (m)':<20} | {'Số ngẫu nhiên (r)':<20} | {'Particle được chọn (chỉ số)':<30} | {'Vị trí x_t_resampled':<20}")
print("-"*100)

for m in range(M):
    # Lấy số ngẫu nhiên r từ danh sách đã cung cấp
    r_m = resampling_uniform_samples[m]
    
    # Tìm particle được chọn (Roulette Wheel)
    # np.searchsorted tìm chỉ số đầu tiên mà giá trị r_m có thể được chèn vào cdf_weights
    # mà vẫn duy trì thứ tự sắp xếp.
    chosen_particle_index = np.searchsorted(cdf_weights, r_m)
    
    # Đảm bảo chỉ số không vượt quá giới hạn (có thể xảy ra nếu r_m = 1.0 và cdf cuối cùng < 1.0 do sai số)
    if chosen_particle_index >= M:
        chosen_particle_index = M - 1
        
    particles_t_resampled[m] = particles_t_predicted[chosen_particle_index]
    
    print(f"{m+1:<20} | {r_m:<20.4f} | {chosen_particle_index+1:<30} | {particles_t_resampled[m]:<20.4f}")



=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
BƯỚC TÁI LẤY MẪU (RESAMPLING)
Particle mới (m)     | Số ngẫu nhiên (r)    | Particle được chọn (chỉ số)    | Vị trí x_t_resampled
----------------------------------------------------------------------------------------------------
1                    | 0.0500               | 2                              | 2.9600              
2                    | 0.1200               | 2                              | 2.9600              
3                    | 0.1800               | 4                              | 2.9200              
4                    | 0.2300               | 4                              | 2.9200              
5                    | 0.2900               | 6                              | 2.8800              
6                    | 0.3300               | 6                              | 2.8800              
7                    | 0.3900               | 8                              | 2.8400              
8       


## Phần 4: Kết quả cuối cùng


In [17]:


print("\n="*30)
print("KẾT QUẢ PARTICLE FILTER SAU MỘT BƯỚC")
print("="*30)

print("\n--- Các particle dự đoán và trọng số của chúng (trước resampling) ---")
print(f"{'Particle (m)':<15} | {'x_t_predicted':<20} | {'Weight w_t (chưa chuẩn hóa)':<30} | {'Normalized Weight':<20}")
print("-"*95)
for m in range(M):
    print(f"{m+1:<15} | {particles_t_predicted[m]:<20.4f} | {weights_t[m]:<30.6e} | {normalized_weights_t[m]:<20.6f}")

print("\n--- Các particle sau khi tái lấy mẫu (resampling) ---")
print(f"{'Particle mới (m)':<20} | {'x_t_resampled':<20}")
print("-"*45)
for m in range(M):
    print(f"{m+1:<20} | {particles_t_resampled[m]:<20.4f}")




=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
KẾT QUẢ PARTICLE FILTER SAU MỘT BƯỚC

--- Các particle dự đoán và trọng số của chúng (trước resampling) ---
Particle (m)    | x_t_predicted        | Weight w_t (chưa chuẩn hóa)    | Normalized Weight   
-----------------------------------------------------------------------------------------------
1               | 3.0200               | 1.941861e+00                   | 0.048527            
2               | 2.9600               | 3.332246e+00                   | 0.083273            
3               | 3.0600               | 1.109208e+00                   | 0.027719            
4               | 2.9200               | 3.910427e+00                   | 0.097721            
5               | 3.1000               | 5.399097e-01                   | 0.013492            
6               | 2.8800               | 3.910427e+00                   | 0.097721            
7               | 3.1400               | 2.239453e-01                 

Nhận xét (ví dụ):

- Các particle có trọng số cao (normalized weight lớn) sẽ có khả năng được chọn lại nhiều lần
  trong bước tái lấy mẫu.
  
- Các particle có trọng số thấp có thể sẽ bị loại bỏ.

- Tập particle sau tái lấy mẫu sẽ tập trung hơn quanh các vùng có xác suất cao (phù hợp với phép đo).